# Lección 01 — Tu Primer Agente de IA con Claude

En este notebook vas a construir tu primer agente paso a paso.

El agente que vamos a crear es un **recomendador de destinos de viaje**. Le vas a decir qué tipo de viaje buscás y él va a consultar una lista de destinos disponibles para darte la mejor recomendación.

### Lo que vas a aprender:
1. Cómo conectarte a Claude con la API de Anthropic
2. Cómo darle herramientas (tools) a un agente
3. Cómo funciona el bucle agente (razonar → actuar → observar)
4. Cómo ejecutar tu agente con un mensaje real

## Paso 1 — Instalación

Primero instalamos las librerías necesarias. Solo necesitás dos:
- `anthropic` — la librería oficial de Anthropic para usar Claude
- `python-dotenv` — para leer tu API key de forma segura desde un archivo `.env`

In [ ]:
%pip install anthropic python-dotenv -q

## Paso 2 — Configurar tu API Key

Para usar Claude necesitás una API key. Si todavía no tenés una:
1. Creá una cuenta en [console.anthropic.com](https://console.anthropic.com/)
2. Andá a **API Keys** y creá una nueva
3. Creá un archivo `.env` en la carpeta raíz del curso con este contenido:

```
ANTHROPIC_API_KEY=tu_api_key_aqui
```

> **Importante:** Nunca compartas tu API key ni la subas a GitHub. El archivo `.env` está en el `.gitignore` para protegerla.

In [ ]:
import anthropic
import json
import os
from dotenv import load_dotenv

# Cargar las variables de entorno desde el archivo .env
load_dotenv()

# Crear el cliente de Anthropic — usa automáticamente tu ANTHROPIC_API_KEY
client = anthropic.Anthropic()

print("Conexión con Claude establecida correctamente.")

## Paso 3 — Definir una Herramienta (Tool)

Una **herramienta** es una función de Python que el agente puede llamar cuando la necesita.

En este caso, la herramienta `obtener_destinos` devuelve una lista de destinos disponibles. El agente la va a usar cuando el usuario le pida una recomendación.

Para que Claude sepa cuándo y cómo usar la herramienta, necesitamos describírsela con un schema JSON.

In [ ]:
# Esta es la función Python que el agente puede llamar
def obtener_destinos():
    """Devuelve una lista de destinos de viaje disponibles."""
    return [
        {"nombre": "Barcelona", "clima": "mediterráneo", "tipo": "ciudad costera"},
        {"nombre": "París", "clima": "templado", "tipo": "ciudad cultural"},
        {"nombre": "Cancún", "clima": "tropical", "tipo": "playa"},
        {"nombre": "Buenos Aires", "clima": "templado", "tipo": "ciudad cosmopolita"},
        {"nombre": "Tokio", "clima": "variado", "tipo": "ciudad tecnológica"},
        {"nombre": "Cartagena", "clima": "cálido", "tipo": "playa colonial"},
        {"nombre": "Ciudad de México", "clima": "templado", "tipo": "ciudad cultural"},
        {"nombre": "Bali", "clima": "tropical", "tipo": "naturaleza y playa"},
    ]

# Así le describimos la herramienta a Claude (schema JSON)
herramientas = [
    {
        "name": "obtener_destinos",
        "description": "Obtiene la lista completa de destinos de viaje disponibles con información sobre clima y tipo de experiencia.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
]

print("Herramienta 'obtener_destinos' definida.")

## Paso 4 — El Bucle Agente

Aquí está el corazón de todo agente. El bucle funciona así:

1. **Razonar:** Claude recibe el mensaje del usuario y decide qué hacer
2. **Actuar:** Si necesita información, llama a una herramienta
3. **Observar:** Recibe el resultado de la herramienta
4. **Repetir** hasta tener una respuesta final para el usuario

```
Usuario → Claude → ¿Necesito una tool? → Sí → Llamar tool → Resultado → Claude → Respuesta
                                        → No → Respuesta directa
```

In [ ]:
def ejecutar_agente(pregunta_usuario):
    """Ejecuta el bucle completo del agente y devuelve la respuesta final."""
    
    print(f"Usuario: {pregunta_usuario}")
    print("-" * 50)
    
    # Historial de mensajes — empieza con la pregunta del usuario
    mensajes = [
        {"role": "user", "content": pregunta_usuario}
    ]
    
    # Instrucciones del agente (system prompt)
    system_prompt = """Eres un agente experto en recomendaciones de viaje para latinoamericanos. 
    Usá la herramienta obtener_destinos para ver los destinos disponibles y luego recomendá 
    el más adecuado según las preferencias del usuario. Respondé siempre en español, 
    de forma amigable y con detalles útiles."""
    
    # Bucle agente: se repite hasta obtener una respuesta final
    while True:
        # Llamar a Claude
        respuesta = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=1024,
            system=system_prompt,
            tools=herramientas,
            messages=mensajes
        )
        
        # ¿Claude quiere usar una herramienta?
        if respuesta.stop_reason == "tool_use":
            # Encontrar qué herramienta quiere usar
            uso_tool = next(b for b in respuesta.content if b.type == "tool_use")
            print(f"Agente usa herramienta: {uso_tool.name}")
            
            # Ejecutar la herramienta correspondiente
            if uso_tool.name == "obtener_destinos":
                resultado_tool = obtener_destinos()
            
            print(f"Resultado: {len(resultado_tool)} destinos encontrados")
            print("-" * 50)
            
            # Agregar la respuesta de Claude y el resultado de la tool al historial
            mensajes.append({"role": "assistant", "content": respuesta.content})
            mensajes.append({
                "role": "user",
                "content": [{
                    "type": "tool_result",
                    "tool_use_id": uso_tool.id,
                    "content": json.dumps(resultado_tool, ensure_ascii=False)
                }]
            })
            
            # Continuar el bucle para que Claude procese el resultado
            continue
        
        # Si no usa herramienta, tenemos la respuesta final
        respuesta_final = next(b.text for b in respuesta.content if b.type == "text")
        print(f"Agente: {respuesta_final}")
        return respuesta_final

## Paso 5 — Probá el Agente

Ahora viene lo divertido. Ejecutá la celda de abajo para ver a tu agente en acción.

In [ ]:
# Probá con diferentes preguntas:
ejecutar_agente("Busco un destino de playa cálido, preferiblemente en Latinoamérica. ¿Qué me recomendás?")

In [ ]:
# Otra consulta diferente:
ejecutar_agente("Quiero conocer una ciudad cultural con buena gastronomía. Tengo presupuesto medio.")

## Resumen de lo que aprendiste

En esta lección construiste tu primer agente de IA. Esto es lo que pasó por detrás:

1. **Conectaste Claude** usando la API de Anthropic con `anthropic.Anthropic()`
2. **Definiste una herramienta** (`obtener_destinos`) que el agente puede llamar
3. **Implementaste el bucle agente:**
   - Claude razona y decide llamar a la herramienta
   - La herramienta ejecuta y devuelve datos
   - Claude procesa los datos y genera la respuesta final

### Conceptos clave:
- `tools` — lista de herramientas disponibles para el agente
- `stop_reason == "tool_use"` — Claude quiere usar una herramienta
- `tool_result` — así le devolvés el resultado al agente
- `system` — las instrucciones que definen el comportamiento del agente

---

En la **Lección 02** vamos a explorar frameworks que simplifican este bucle y permiten construir agentes mucho más complejos.